In [1]:
!pip install -q kaggle transformers accelerate librosa soundfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
cupy-cuda12x 13.1.0 requires numpy<1.29,>=1.22, but you have numpy 2.0.2 which is incompatible.
tensorflow 2.15.1 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.0.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m pip install --user "numpy<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 15.5 MB/s  0:00:01m0:00:010:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os

# Kaggle-Zugangsdaten hier eintragen
os.environ["KAGGLE_USERNAME"] = "DEIN_USERNAME"
os.environ["KAGGLE_KEY"]      = "DEIN_API_KEY"

# Kaggle-Dataset-Slug (den Teil der URL nach kaggle.com/datasets/)
DATASET_SLUG = "piyushagni5/berlin-database-of-emotional-speech-emodb"
DOWNLOAD_DIR = "/tmp/emodb"

# Prompt-Modus: 'normal' oder 'prosodic'
PROMPT_MODE = "normal"

# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

Dataset URL: https://www.kaggle.com/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb
License(s): copyright-authors
100%|██████████████████████████████████████| 38.0M/38.0M [00:01<00:00, 25.5MB/s]



In [3]:
from pathlib import Path
all_files = sorted(Path(DOWNLOAD_DIR).rglob("*.wav"))
print(f"Gefundene WAV-Dateien: {len(all_files)}")
for f in all_files[:5]:
    print(f"  {f}")

Gefundene WAV-Dateien: 535
  /tmp/emodb/wav/03a01Fa.wav
  /tmp/emodb/wav/03a01Nc.wav
  /tmp/emodb/wav/03a01Wa.wav
  /tmp/emodb/wav/03a02Fc.wav
  /tmp/emodb/wav/03a02Nc.wav


In [4]:
import json
import torch
import librosa
from transformers import AutoConfig, AutoModelForSpeechSeq2Seq, AutoProcessor

PROMPTS = {
    "normal": (
        "Listen to this audio clip and classify the emotion of the speaker. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
    "prosodic": (
        "Listen to this audio clip and classify the emotion of the speaker "
        "based only on prosodic and acoustic features such as pitch, "
        "speech rate, rhythm and loudness. "
        "Ignore the spoken words and linguistic content entirely. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
}

PROMPT = PROMPTS[PROMPT_MODE]
output_file = f"emotion_results_meralion_{PROMPT_MODE}.json"

# Resume
results = {}
if os.path.exists(output_file):
    with open(output_file) as f:
        results = json.load(f)
    print(f"{len(results)} Dateien bereits verarbeitet")

# Modell laden (gated repo — setzt HF_TOKEN via huggingface_hub login voraus).
# MERaLiON-3 registriert sich als AutoModelForSpeechSeq2Seq via auto_map.
#
# Workaround: In transformers 5.x wirft der Zugriff auf ein nicht gesetztes
# Config-Feld AttributeError (statt None zurueckzugeben wie in 4.x). MERaLiONs
# modeling_meralion3.py erwartet aber config.pad_token_id. Wir injizieren den
# Wert aus der text_config (Gemma2).
print("Lade Modell...")
MODEL_ID = "MERaLiON/MERaLiON-3-10B-preview"

cfg = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
cfg.pad_token_id = getattr(cfg.text_config, "pad_token_id", None)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    config=cfg,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
).eval()

# Zielsamplerate fuer das Audio — MERaLiON erwartet 16 kHz
SR = 16000
print(f"Modell geladen auf: {next(model.parameters()).device}")

2026-04-22 23:47:36.315476: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 23:47:36.344621: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-22 23:47:36.344640: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 23:47:36.345563: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-22 23:47:36.350456: I tensorflow/core/platform/cpu_feature_guar

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in st

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/sw/ubuntu22/generic/python/3.11.8_dir/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in st

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

In [ ]:
  import sys
  !{sys.executable} -m pip install --user --upgrade --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128 

In [ ]:
  import torch
  print("Device    :", torch.cuda.get_device_name(0))                                                                                                                                                                                                                           
  print("Capability:", torch.cuda.get_device_capability(0))   # e.g. (9, 0) for H100
  print("Arch list :", torch.cuda.get_arch_list())            # archs torch was compiled for                                                                                                                                                                                    
  print("torch     :", torch.__version__, "CUDA:", torch.version.cuda)

In [ ]:
total_files = len(all_files)

# MERaLiON-3 Chat-Template: der Platzhalter <SpeechHere> wird vom Processor
# durch die Audio-Features ersetzt. Falls die Model-Card ein anderes Format
# zeigt, hier anpassen.
def build_conversation(prompt):
    return [{
        "role": "user",
        "content": f"Given the following audio: <SpeechHere>\n\n{prompt}",
    }]

for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Uebersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        audio, _ = librosa.load(str(audio_file), sr=SR)

        conversation = build_conversation(PROMPT)
        text = processor.tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )

        inputs = processor(text=text, audios=[audio], sampling_rate=SR, return_tensors="pt")
        # Float-Tensoren (Audio-Features) muessen in das Modell-dtype (bf16)
        # gecastet werden; int-Tensoren (input_ids, attention_mask) bleiben.
        inputs = {
            k: (v.to(model.device, dtype=torch.bfloat16) if v.is_floating_point()
                else v.to(model.device))
            for k, v in inputs.items()
        }

        with torch.no_grad():
            generate_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        generate_ids = generate_ids[:, inputs["input_ids"].size(1):]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateien verarbeitet.")

In [ ]:
import pandas as pd

df = pd.DataFrame(list(results.items()), columns=["file", "emotion"])
print(df["emotion"].value_counts().to_string())

csv_file = output_file.replace(".json", ".csv")
df.to_csv(csv_file, index=False)
print(f"\nCSV gespeichert: {csv_file}")